# 지자기 지문 매칭 개발

기구별 지자기 벡터(mag_x, mag_y, mag_z)를 코사인 유사도로 비교하여
90% 이상 식별 정확도를 달성하는 매칭 임계값을 탐색한다.

확정된 상수(`MATCH_THRESHOLD`, `MIN_SETTLE_SAMPLES`)는
`backend/pipeline/mag_fingerprint.py`에 이식된다.

## 1. 기구별 지자기 지문 정의

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

# macOS 한글 폰트 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

# 기구별 지자기 벡터 (μT)
# 일부 기구를 의도적으로 유사하게 설계 — 실제 헬스장에서 인접한 기구의 자기장이
# 비슷할 수 있는 상황을 모사 (레그컬 ↔ 레그익스텐션, 랫풀다운 ↔ 스미스머신)
EQUIPMENT_FINGERPRINTS = {
    '레그프레스':      np.array([ 30.5, -15.2,  45.1]),
    '랫풀다운':        np.array([-10.0,  38.0,  20.0]),   # 스미스머신과 유사
    '스미스머신':      np.array([-12.0,  42.0,  18.0]),   # 랫풀다운과 유사
    '펙덱플라이':      np.array([  5.1, -38.6,  60.3]),
    '레그컬':          np.array([-40.0,  25.0,  10.5]),   # 레그익스텐션과 유사
    '레그익스텐션':    np.array([-38.0,  28.0,   8.0]),   # 레그컬과 유사
}

# 기구 간 코사인 유사도 확인
names = list(EQUIPMENT_FINGERPRINTS.keys())
vecs  = [v / np.linalg.norm(v) for v in EQUIPMENT_FINGERPRINTS.values()]

print('기구 간 코사인 유사도 (1.0에 가까울수록 혼동 위험):')
for (i, n1), (j, n2) in combinations(enumerate(names), 2):
    sim = float(np.dot(vecs[i], vecs[j]))
    flag = ' ← 유사' if sim > 0.95 else ''
    print(f'  {n1} ↔ {n2}: {sim:.4f}{flag}')

## 2. 노이즈 포함 테스트 샘플 생성

In [ ]:
NOISE_STD      = 8.0   # μT — 벡터 크기(~50μT) 대비 약 16%, 오인식이 발생하는 현실적 수준
SAMPLES_PER_EQ = 50

test_samples = []
for name, vec in EQUIPMENT_FINGERPRINTS.items():
    for _ in range(SAMPLES_PER_EQ):
        noisy = vec + np.random.normal(0, NOISE_STD, 3)
        test_samples.append((name, noisy))

print(f'총 테스트 샘플: {len(test_samples)}개  (기구당 {SAMPLES_PER_EQ}개, 노이즈 std={NOISE_STD} μT)')

## 3. 코사인 유사도 기반 매칭 구현

In [ ]:
import math

def cosine_similarity(a, b):
    dot    = float(np.dot(a, b))
    norm_a = float(np.linalg.norm(a))
    norm_b = float(np.linalg.norm(b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def match(query_vec, fingerprints: dict, threshold: float):
    best_name, best_score = None, -1.0
    for name, fp_vec in fingerprints.items():
        score = cosine_similarity(query_vec, fp_vec)
        if score > best_score:
            best_score, best_name = score, name
    return best_name if best_score >= threshold else None, best_score

## 4. 그리드 탐색 — 임계값 vs 정확도

In [ ]:
thresholds = np.arange(0.80, 1.00, 0.01)
results = []

for thresh in thresholds:
    correct = 0
    unknown = 0
    wrong   = 0
    for true_name, noisy_vec in test_samples:
        predicted, score = match(noisy_vec, EQUIPMENT_FINGERPRINTS, thresh)
        if predicted is None:
            unknown += 1
        elif predicted == true_name:
            correct += 1
        else:
            wrong += 1
    total = len(test_samples)
    results.append({
        'threshold': thresh,
        'accuracy': correct / total,
        'unknown_rate': unknown / total,
        'wrong_rate': wrong / total,
    })

import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False, float_format='{:.4f}'.format))

plt.figure(figsize=(10, 4))
plt.plot(df['threshold'], df['accuracy'],    label='정확도 (correct)')
plt.plot(df['threshold'], df['unknown_rate'], label='미인식률 (unknown)', linestyle='--')
plt.plot(df['threshold'], df['wrong_rate'],   label='오인식률 (wrong)',   linestyle=':')
plt.axvline(0.97, color='red', linestyle='--', label='채택값 (0.97)')
plt.axhline(0.90, color='gray', linestyle=':', alpha=0.5, label='목표 정확도 90%')
plt.xlabel('threshold')
plt.title('코사인 유사도 임계값 vs 식별 성능')
plt.legend()
plt.tight_layout()
plt.show()

## 5. MIN_SETTLE_SAMPLES 탐색

단일 샘플 대신 N개 평균 벡터로 매칭하면 노이즈 억제 효과가 있다.

In [ ]:
THRESHOLD = 0.97
sample_sizes = [1, 3, 5, 10, 20]

for n_samples in sample_sizes:
    correct = 0
    for true_name, fp_vec in EQUIPMENT_FINGERPRINTS.items():
        # n_samples개 평균 벡터 생성
        batch = fp_vec + np.random.normal(0, NOISE_STD, (n_samples, 3))
        avg   = batch.mean(axis=0)
        predicted, _ = match(avg, EQUIPMENT_FINGERPRINTS, THRESHOLD)
        if predicted == true_name:
            correct += 1
    acc = correct / len(EQUIPMENT_FINGERPRINTS)
    print(f'n_samples={n_samples:2d}  기구별 1회 테스트 정확도: {acc:.2f}')

## 6. 상수 확정

### 탐색 결과 요약

| threshold | accuracy | unknown_rate | wrong_rate |
|-----------|----------|--------------|------------|
| 0.80 | 0.7333 | 0.0000 | **0.2667** |
| 0.90 | 0.7233 | 0.0200 | **0.2567** |
| 0.95 | 0.6533 | 0.1167 | **0.2300** |
| 0.97 | 0.5333 | 0.2733 | **0.1933** |
| 0.99 | 0.2933 | 0.5833 | **0.1233** |

### 발견된 한계

어떤 임계값에서도 **오인식률이 0%에 도달하지 않는다.**

원인: 의도적으로 유사하게 설계한 두 벡터쌍(랫풀다운↔스미스머신, 레그컬↔레그익스텐션)이
NOISE_STD=8.0 μT 하에서 서로 혼동된다. 임계값을 높이면 오인식이 줄지만
미인식이 늘어나는 트레이드오프가 발생한다.

이는 **데이터 품질 문제**이지 임계값 선택 문제가 아니다.
실제 헬스장에서 기구 간 거리와 철제 구조 차이에 따라 자기장이 충분히 달라지면
오인식률은 크게 낮아질 수 있다.

### 임계값 선택 근거

이 프로젝트에서 오인식(A 기구를 B 기구로 표시)은 미인식(사용자에게 이름 입력 요청)보다
훨씬 나쁜 사용자 경험이다. 따라서 **오인식률 최소화를 우선**한다.

- threshold=0.97: 오인식률이 처음으로 0.20 아래로 내려가는 변곡점
- 미인식 시 `equipment_unknown` 이벤트로 사용자 입력을 유도하므로 허용 가능

**MATCH_THRESHOLD = 0.97** 채택. 단, 실제 센서 데이터로 재검증 필요.

### MIN_SETTLE_SAMPLES

n_samples=5 이상에서 샘플 평균이 노이즈를 효과적으로 억제함을 확인.
**MIN_SETTLE_SAMPLES = 5** 유지.

In [ ]:
MATCH_THRESHOLD   = 0.97
MIN_SETTLE_SAMPLES = 5
print(f'MATCH_THRESHOLD    = {MATCH_THRESHOLD}')
print(f'MIN_SETTLE_SAMPLES = {MIN_SETTLE_SAMPLES}')